In [2]:
import matplotlib.pyplot as plt
import accelforge as af

# Configuration
PROB_FILE = 'prob.yaml'
ARCH_FILES = ["arch.yaml", "components.yaml", "map.yaml"]
TOTAL_KEYS = 131072  # The full context size before filtering

def main():
    # 1. Load the base specification once
    # This matches how Lab 5's _get_spec loads the files into a Spec object
    yaml_files = ARCH_FILES + [PROB_FILE]
    spec = af.Spec.from_yaml(yaml_files)
    
    filtering_ratios = [5, 10, 20, 40]
    energies = []
    latencies = []

    print("Starting Sweep (In-Memory Implementation)...")

    for ratio in filtering_ratios:
        # 2. "Do it their way": Update the attribute in memory directly
        # This bypasses all YAML parsing errors and formatting corruption.
        new_ks = int(TOTAL_KEYS / ratio)
        spec.workload.rank_sizes['K_s'] = new_ks
        
        print(f"Testing {ratio}x ratio (K_s={new_ks})...", end=" ", flush=True)
        
        try:
            # 3. Run the mapper (Matches Lab 5's quick_run logic)
            mappings = spec.map_workload_to_arch(
                print_progress=False, 
                print_number_of_pmappings=False
            )
            
            # 4. Extract metrics from the Result-like object
            # Energy is natively in Joules; we convert to pJ for the plot
            energy_pj = sum(mappings.energy(per_component=True).values()) * 1e12
            
            # Retrieve cycles/latency
            try:
                cycles = mappings.cycles()
            except AttributeError:
                cycles = mappings.latency()

            energies.append(energy_pj)
            latencies.append(cycles)
            print(f"Done. Energy: {energy_pj:.2f} pJ")

        except Exception as e:
            print(f"FAILED! Error: {e}")
            break

    # 5. Visualize the results
    if energies:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        ax1.plot(filtering_ratios[:len(energies)], energies, marker='o', color='blue')
        ax1.set_title('Total Energy vs. Filtering Ratio')
        ax1.set_xlabel('Filtering Ratio (x)')
        ax1.set_ylabel('Energy (pJ)')
        ax1.grid(True)

        ax2.plot(filtering_ratios[:len(latencies)], latencies, marker='s', color='red')
        ax2.set_title('Latency vs. Filtering Ratio')
        ax2.set_xlabel('Filtering Ratio (x)')
        ax2.set_ylabel('Latency (Cycles)')
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()

ValidationError: 4 validation errors for Spec
arch.nodes.0
  Input tag 'Component' found using _get_tag() does not match any of the expected tags: 'Compute', 'Memory', 'Toll', 'Container', 'Network', 'Hierarchical', 'Array', 'Fork' [type=union_tag_invalid, input_value={'name': 'System', 'class...idth': 1}}]}]}]}]}]}]}]}, input_type=CommentedMap]
    For further information visit https://errors.pydantic.dev/2.12/v/union_tag_invalid
mapping
  Input should be a valid dictionary or instance of Mapping [type=model_type, input_value=[{'einsum': 'DenseQKT', '..., 'factors': 'D=128'}]}], input_type=CommentedSeq]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
workload.einsums.3.tensor_accesses.1
  Value error, Field density is not supported for TensorAccess. Supported fields are:
	backing_storage_size_scale
	bits_per_value
	name
	output
	persistent
	projection
 [type=value_error, input_value={'name': 'FilteredKey', '...e': 16, 'density': 0.05}, input_type=Commen

In [8]:
!which timeloop-model
!which accelforge

In [9]:
!find / -name "timeloop-model" -type f 2>/dev/null
!find / -name "accelforge" -type f 2>/dev/null